# RUN_BATCH — Orchestrator batch analisa PV per tanggal

Menjalankan template notebook (mis. `20260209stringmap_v1.5.ipynb`) untuk
banyak tanggal sekaligus **tanpa mengubah file template**.

**Cara pakai:**
1. Upload notebook ini ke Drive / buka di Google Colab.
2. Edit sel **Config** di bawah: `TEMPLATE_NB` dan isi `DATE_TO_URL`.
3. **Runtime > Run all**. Hasil utama tersimpan otomatis ke `Cek PV String/outputs/`.

Catatan: butuh runtime Colab (memakai `google.colab` + Drive). Folder harian
harus ter-share "anyone with link" supaya gdown bisa mengunduh.

In [ ]:
# ============================ CONFIG (EDIT DI SINI) ============================
# Lokasi repo di Google Drive (sudah sesuai notebook Anda):
REPO_DIR     = '/content/drive/MyDrive/Cek PV String'

# Lokasi file template .ipynb di Drive (GANTI sesuai lokasi Anda):
TEMPLATE_NB  = '/content/drive/MyDrive/Cek PV String/notebook/20260209stringmap_v1.5.ipynb'

SKIP_IF_DONE = True   # True = lewati tanggal yang m2_findings_YYYYMMDD.xlsx-nya sudah ada
QUIET_PLOTS  = True   # True = jangan render heatmap saat batch (file output TIDAK terpengaruh)

# Peta: 'YYYY-MM-DD' -> URL folder Google Drive harian (share: anyone with link).
# Tanggal hanya dipakai utk skip/log; tanggal output tetap auto-detect dari data.
DATE_TO_URL = {
    '2026-02-09': 'https://drive.google.com/drive/folders/17JX8ZaH0Gh24e_8DUKqKS9Rhpzvjx0IL',
    # '2026-02-10': 'https://drive.google.com/drive/folders/XXXX',
    # ... tambahkan sesuai range tanggal Anda ...
}
# =============================================================================

In [ ]:
# ============================ ENGINE (biarkan apa adanya) =====================
from google.colab import drive
drive.mount('/content/drive')

import json, os, sys, gc

if QUIET_PLOTS:
    import matplotlib
    matplotlib.use('Agg')        # batch: tidak render ke layar (file output tidak berubah)
import matplotlib.pyplot as plt

assert os.path.isdir(REPO_DIR),    f'REPO_DIR tidak ditemukan: {REPO_DIR}'
assert os.path.isfile(TEMPLATE_NB), f'TEMPLATE_NB tidak ditemukan: {TEMPLATE_NB}'

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Nonaktifkan unduhan browser pada Cell 8 template (TANPA mengubah file template)
import google.colab.files as _gf
_gf.download = lambda *a, **k: print('   [skip files.download]', *a)

# Muat template sekali; ambil hanya code cell secara berurutan
_nb = json.load(open(TEMPLATE_NB, encoding='utf-8'))
CODE_CELLS = [''.join(c['source']) for c in _nb['cells'] if c['cell_type'] == 'code']
OUT_DIR = os.path.join(REPO_DIR, 'outputs')

def _already_done(datestr):
    ymd = datestr.replace('-', '')
    return os.path.isfile(os.path.join(OUT_DIR, f'm2_findings_{ymd}.xlsx'))

def _inject_url(src, url):
    out = []
    for line in src.split('\n'):
        if line.lstrip().startswith('DRIVE_FOLDER_URL') and '=' in line:
            indent = line[:len(line) - len(line.lstrip())]
            out.append(f'{indent}DRIVE_FOLDER_URL = "{url}"')
        else:
            out.append(line)
    return '\n'.join(out)

def run_one_day(url):
    g = {'__name__': '__main__'}          # namespace baru tiap hari (anti bocor state)
    for src in CODE_CELLS:
        exec(_inject_url(src, url), g)

ok = skip = fail = 0
for datestr, url in DATE_TO_URL.items():
    if SKIP_IF_DONE and _already_done(datestr):
        print(f'SKIP {datestr}: output sudah ada'); skip += 1; continue
    print(f'==================  {datestr}  ==================')
    try:
        run_one_day(url); print(f'OK   {datestr}'); ok += 1
    except Exception as e:
        print(f'FAIL {datestr}: {type(e).__name__}: {e}'); fail += 1
    finally:
        plt.close('all'); gc.collect()

print(f'\n=== RINGKASAN ===  sukses={ok}  skip={skip}  gagal={fail}  total={len(DATE_TO_URL)}')
print(f'Hasil: {OUT_DIR}/m2_findings_YYYYMMDD.xlsx (+ .jsonl, pr_daily_*.csv, baseline/)')